## Comparing IMF models

#### NOTE: take outside of "notebooks" before running, there is still a heirchy bug 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from matplotlib.lines import Line2D
%matplotlib inline

# Import the custom package we designed
from pop3_cmb.cosmology import CAMBRunner
from pop3_cmb.faraday import FaradayModel
from pop3_cmb.spectrum import VVSpectrum
from pop3_cmb.config import CONSTANTS, COSMOLOGY, PARAMS

# Plotting settings for publication-quality figures
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.family'] = 'STIXGeneral'
plt.rcParams['font.size'] = 16
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded successfully.")

In [ ]:
camb_run = CAMBRunner(COSMOLOGY, PARAMS['k_min'], PARAMS['k_max'], PARAMS['n_k_points'], PARAMS['z_min'], PARAMS['z_max'])
z_grid, k_grid, Plin = camb_run.get_linear_matter_power()
r_grid = camb_run.get_comoving_distance(z_grid)
ls_ee, Cl_EE = camb_run.get_primordial_Cl_EE(lmax=3000)

In [ ]:
# layout models: 

# freq (Hz)

freqs = np.array([27, 1.0, 0.5, 0.8, 0.9]) * 10 ** 9
freqs = list(freqs)

# (M_low, M_high)
ranges = [(10,100), (10,300), (10,1000)]

# Model, gamma, alpha, beta, Mch, sigma 
IMFModel = [('FlatIMF', None, None, None, None, None),
        ('PowerIMF', 1.35, None, None, None, None), # Salpeter
        ('LarsonIMF', None, None, None, None, None),
        ('ChabrierIMF', None, 2.35, 1.8, 20, None), # sharper cutoff below Mch than Larson's beta = 1
        ('LogNormalIMF', None, None, None, 20, 0.5)]

In [ ]:
for freq in freqs:
    for rang in ranges:
        for IMFs in IMFModel:
            model, gamma, alpha, beta, Mch, sigma = IMFs

            fm = FaradayModel(camb_run, PARAMS, CONSTANTS, k_grid, Plin, 
                            freq = freq,
                            Model = model,
                            M_low = rang[0],
                            M_high = rang[1],
                            gamma = gamma, 
                            alpha = alpha, 
                            beta = beta, 
                            Mch = Mch, 
                            sigma = sigma)

            P_alpha = fm.compute_P_alpha()
            convolution = VVSpectrum(2, PARAMS['ell_max'], P_alpha, z_grid, r_grid, k_grid, Cl_EE)
            ls_vv, Cl_VV = convolution.compute_Cl_VV()
            np.savetxt(f'model_results/Cl_VV_freq{str(freq / 10 ** 9)}_model{model}_Mlow{str(rang[0])}_Mhigh{str(rang[1])}_gamma{str(gamma)}_alpha{str(alpha)}_beta{str(beta)}_Mch{str(Mch)}_sigma{str(sigma)}.txt', np.column_stack((ls_vv, Cl_VV)), header='ell Cl_VV', comments="")

In [ ]:
# unwrap nrap files (place this in external workflow file for use)

from pathlib import Path
import pandas as pd
import numpy as np
import re

def convert_value(value):
    if value == "None":
        return None
    
    try:
        if "." in value or "e" in value.lower():
            return float(value)
        return int(value)
    except ValueError:
        return value


def unwrap_filename(filename):
    """
    Extract model parameters from filename   
    """

    name = Path(filename).name

    pattern = (
        r"Cl_VV_"
        r"freq(?P<freq>.*?)_"
        r"model(?P<model>.*?)_"
        r"Mlow(?P<Mlow>.*?)_"
        r"Mhigh(?P<Mhigh>.*?)_"
        r"gamma(?P<gamma>.*?)_"
        r"alpha(?P<alpha>.*?)_"
        r"beta(?P<beta>.*?)_"
        r"Mch(?P<Mch>.*?)_"
        r"sigma(?P<sigma>.*?)"
        r"\.txt$"
    )

    match = re.match(pattern, name)

    if match is None:
        raise ValueError(f"Filename does not match expected format: {name}")

    values = match.groupdict()

    for key in values:
        if key != "model":
            values[key] = convert_value(values[key])

    values["filename"] = name

    return values


def unwrap_folder(folder_path):
    folder_path = Path(folder_path)

    rows = []

    for file in folder_path.glob("Cl_VV_*.txt"):
        rows.append(unwrap_filename(file))

    df = pd.DataFrame(rows)

    df = df[
        [
            "filename",
            "freq",
            "model",
            "Mlow",
            "Mhigh",
            "gamma",
            "alpha",
            "beta",
            "Mch",
            "sigma",
        ]
    ]

    return df

def label(rowNumber, df):
    row = df.iloc[rowNumber]

    non_none_values = (
        row
        .drop(labels=["filename"])
        .replace("None", np.nan)
        .dropna()
    )

    label = ", ".join([f"{key}={value}" for key, value in non_none_values.items()])

    return label

In [ ]:
df = unwrap_folder('model_results')
df.head()

In [ ]:
row = df.iloc[5]

non_none_values = (
    row
    .drop(labels=["filename"])
    .replace("None", np.nan)
    .dropna()
)

print(non_none_values)

label = ", ".join([f"{key}={value}" for key, value in non_none_values.items()])

print(label)

In [ ]:
# plotting

folder_path = Path('model_results')
for count, file in enumerate(folder_path.glob("Cl_VV_*.txt")):
    dfVV = pd.read_csv(file, sep=r"\s+")
    ls, Cl_VV = dfVV['ell'], dfVV['Cl_VV']

    row = df.iloc[count]

    non_none_values = (
        row
        .drop(labels=["filename"])
        .replace("None", np.nan)
        .dropna()
    )

    label = ", ".join([f"{key}={value}" for key, value in non_none_values.items()])

    plt.plot(ls_vv, ls_vv*(ls_vv+1)*Cl_VV/(2*np.pi), 'o-', color='crimson', lw=2,) #label=r'Pop III Signal'
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel(r'Multipole $\ell$')
    plt.ylabel(r'$\ell(\ell+1)\ C_\ell^{VV}\ /\ 2\pi$ [$\mu K^2$]')
    plt.title(label)
    plt.grid(True, which='major', alpha=0.5)
    # plt.savefig('./figures/Cl_VV.pdf', )
    plt.show()